#### Importing Libraries

In [1]:
# Importing core libraries

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings

# Importing model selection utilities

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

# Importing classification models

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier

# Importing evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Importing label encoding

from sklearn.preprocessing import LabelEncoder

# Importing experiment tracking

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Configuring warnings

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


#### Loading the Processed Datasets

In [2]:
# Defining the processed data directory

processed_dir = Path("../data/processed")

# Loading processed training and testing features

X_train = pd.read_parquet(
    processed_dir / "X_train_processed.parquet"
)

X_test = pd.read_parquet(
    processed_dir / "X_test_processed.parquet"
)

# Loading training and testing target labels

y_train = pd.read_parquet(
    processed_dir / "y_train.parquet"
)["severity"]

y_test = pd.read_parquet(
    processed_dir / "y_test.parquet"
)["severity"]

# Displaying dataset dimensions

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

# Displaying target class distribution

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

# Verifying feature and target alignment

print("\nTraining Indices Aligned:", X_train.index.equals(y_train.index))
print("Testing Indices Aligned:", X_test.index.equals(y_test.index))

Training Features: (800, 13)
Testing Features: (200, 13)

Training Target: (800,)
Testing Target: (200,)

Training Target Distribution:
severity
Minor       286
Critical    267
Moderate    247
Name: count, dtype: int64

Testing Target Distribution:
severity
Minor       72
Critical    66
Moderate    62
Name: count, dtype: int64

Training Indices Aligned: True
Testing Indices Aligned: True


#### Observations on Loaded Datasets

- The training and testing datasets contain **800 and 200 records**, respectively, with 13 processed features.
- All three severity classes are represented in both datasets, with relatively balanced distributions.
- Feature and target indices are correctly aligned, confirming that the datasets are ready for model development.

#### Encoding the Target Variable

In [3]:
# Initializing and fitting the target label encoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Displaying class encoding

print("Target Class Mapping:")

for encoded_value, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name}: {encoded_value}")

# Displaying encoded target dimensions

print("\nEncoded Training Target:", y_train_encoded.shape)
print("Encoded Testing Target:", y_test_encoded.shape)

# Displaying encoded class distribution

print("\nEncoded Training Class Distribution:")
print(pd.Series(y_train_encoded).value_counts().sort_index())

print("\nEncoded Testing Class Distribution:")
print(pd.Series(y_test_encoded).value_counts().sort_index())

# Verifying encoding consistency

print("\nTraining Encoding Consistent:",
      np.array_equal(label_encoder.inverse_transform(y_train_encoded), y_train.to_numpy()))

print("Testing Encoding Consistent:",
      np.array_equal(label_encoder.inverse_transform(y_test_encoded), y_test.to_numpy()))

Target Class Mapping:
Critical: 0
Minor: 1
Moderate: 2

Encoded Training Target: (800,)
Encoded Testing Target: (200,)

Encoded Training Class Distribution:
0    267
1    286
2    247
Name: count, dtype: int64

Encoded Testing Class Distribution:
0    66
1    72
2    62
Name: count, dtype: int64

Training Encoding Consistent: True
Testing Encoding Consistent: True


In [4]:
# Defining artifact directories

processed_dir = Path("../data/processed")
preprocessor_dir = Path("../models/preprocessed")

preprocessor_dir.mkdir(parents=True, exist_ok=True)

# Saving encoded targets with original indices

pd.DataFrame(
    {"severity": y_train_encoded},
    index=y_train.index
).to_parquet(processed_dir / "y_train_encoded.parquet")

pd.DataFrame(
    {"severity": y_test_encoded},
    index=y_test.index
).to_parquet(processed_dir / "y_test_encoded.parquet")

# Saving the fitted label encoder

joblib.dump(
    label_encoder,
    preprocessor_dir / "label_encoder.joblib"
)

# Verifying saved artifacts

for file in [
    processed_dir / "y_train_encoded.parquet",
    processed_dir / "y_test_encoded.parquet",
    preprocessor_dir / "label_encoder.joblib"
]:
    print(f"{file.name}: {'Saved' if file.exists() else 'Missing'}")

y_train_encoded.parquet: Saved
y_test_encoded.parquet: Saved
label_encoder.joblib: Saved


#### Observations on Target Encoding

- The three severity classes were encoded numerically: **Critical (0), Minor (1), and Moderate (2)**.
- The encoded training and testing targets retain **800 and 200 records**, respectively.
- The encoded targets and fitted label encoder were saved for consistent use during model development, evaluation, and explainability.

#### Configuring Cross-Validation and Evaluation Metrics

In [5]:
# Configuring stratified cross-validation

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Defining classification evaluation metrics

scoring_metrics = {
    "Accuracy": "accuracy",
    "Precision_Macro": "precision_macro",
    "Recall_Macro": "recall_macro",
    "F1_Macro": "f1_macro"
}

# Displaying cross-validation configuration

print("Cross-Validation Strategy:", cv_strategy)
print("Number of Folds:", cv_strategy.get_n_splits())

print("\nEvaluation Metrics:")

for metric_name, scoring_method in scoring_metrics.items():
    print(f"{metric_name}: {scoring_method}")

# Verifying stratified class distribution across folds

print("\nClass Distribution Across Folds:")

for fold, (_, validation_idx) in enumerate(
    cv_strategy.split(X_train, y_train_encoded),
    start=1
):
    fold_labels = y_train_encoded[validation_idx]

    class_distribution = (
        pd.Series(fold_labels)
        .value_counts(normalize=True)
        .sort_index()
        .round(3)
    )

    print(f"\nFold {fold}:")
    print(class_distribution.to_dict())

Cross-Validation Strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Number of Folds: 5

Evaluation Metrics:
Accuracy: accuracy
Precision_Macro: precision_macro
Recall_Macro: recall_macro
F1_Macro: f1_macro

Class Distribution Across Folds:

Fold 1:
{0: 0.338, 1: 0.356, 2: 0.306}

Fold 2:
{0: 0.338, 1: 0.356, 2: 0.306}

Fold 3:
{0: 0.331, 1: 0.356, 2: 0.312}

Fold 4:
{0: 0.331, 1: 0.356, 2: 0.312}

Fold 5:
{0: 0.331, 1: 0.362, 2: 0.306}


#### Observations on Cross-Validation

- Stratified 5-Fold Cross-Validation was configured with shuffling and `random_state=42` for reproducibility.
- All five folds maintain consistent class distributions, with approximately **33% Critical, 36% Minor, and 31% Moderate** records.
- Macro F1-score was selected as the primary model comparison metric, supported by accuracy, macro precision, and macro recall.

#### Configuring MLflow Experiment Tracking

In [6]:
# Defining MLflow tracking configuration

project_root = Path("..").resolve()

mlflow_db_path = project_root / "mlflow.db"
mlflow_artifact_dir = project_root / "mlartifacts"

mlflow_artifact_dir.mkdir(parents=True, exist_ok=True)

# Configuring MLflow tracking URI

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_db_path.as_posix()}"
)

# Defining experiment name

experiment_name = "Manufacturing Quality Assurance"

# Creating or retrieving the MLflow experiment

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=mlflow_artifact_dir.as_uri()
    )
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(experiment_name)

# Displaying experiment configuration

print("Experiment Name:", experiment_name)
print("Experiment ID:", experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Artifact Directory:", mlflow_artifact_dir)
print("Experiment Status: Ready")

2026/10/20 23:07:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/20 23:07:01 INFO mlflow.store.db.utils: Updating database tables


Experiment Name: Manufacturing Quality Assurance
Experiment ID: 1
Tracking URI: sqlite:///D:/Projects/ML and Data Science/Manufacturing Quality Assurance/mlflow.db
Artifact Directory: D:\Projects\ML and Data Science\Manufacturing Quality Assurance\mlartifacts
Experiment Status: Ready


#### Observations on MLflow Configuration

- MLflow experiment tracking was configured successfully using a local SQLite database.
- The **Manufacturing Quality Assurance** experiment was created with Experiment ID **1**.
- The tracking database and artifact directory are ready to store model experiments, metrics, and trained models.

#### Developing the Baseline Model

In [7]:
# Initializing the baseline classifier

baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Evaluating the baseline using stratified cross-validation

baseline_cv_results = cross_validate(
    estimator=baseline_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

baseline_metrics = {
    metric: baseline_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting the baseline model on the complete training dataset

baseline_model.fit(X_train, y_train_encoded)

# Saving the trained baseline model

trained_models_dir = Path("../models/trained")
trained_models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    baseline_model,
    trained_models_dir / "dummy_classifier.joblib"
)

# Logging the baseline experiment to MLflow

with mlflow.start_run(run_name="Dummy Classifier"):

    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("cv_folds", 5)

    for metric, value in baseline_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "dummy_classifier.joblib"),
        artifact_path="model"
    )

# Creating the model comparison results

model_results = []

model_results.append({
    "Model": "Dummy Classifier",
    **baseline_metrics
})

# Displaying baseline performance

print("Baseline Model: Dummy Classifier")

print("\nCross-Validation Performance:")

for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")
display(pd.DataFrame(model_results))

Baseline Model: Dummy Classifier

Cross-Validation Performance:
Accuracy: 0.3575
Precision_Macro: 0.1192
Recall_Macro: 0.3333
F1_Macro: 0.1756

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Dummy Classifier,0.3575,0.119167,0.333333,0.175566


#### Observations on Baseline Model

- The Dummy Classifier achieved **35.75% accuracy** and a **Macro F1-score of 0.1756**.
- Its low Macro F1-score reflects its majority-class prediction strategy, which fails to identify the other two severity classes.
- These results establish the baseline that subsequent classification models should aim to outperform.

#### Developing the Logistic Regression Model

In [8]:
# Initializing the Logistic Regression classifier

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Evaluating Logistic Regression using stratified cross-validation

logistic_cv_results = cross_validate(
    estimator=logistic_model,
    X=X_train,
    y=y_train_encoded,
    cv=cv_strategy,
    scoring=scoring_metrics,
    n_jobs=1
)

# Calculating mean cross-validation metrics

logistic_metrics = {
    metric: logistic_cv_results[f"test_{metric}"].mean()
    for metric in scoring_metrics
}

# Fitting Logistic Regression on the complete training dataset

logistic_model.fit(X_train, y_train_encoded)

# Saving the trained model

joblib.dump(
    logistic_model,
    trained_models_dir / "logistic_regression.joblib"
)

# Logging the experiment to MLflow

with mlflow.start_run(run_name="Logistic Regression"):

    mlflow.log_params({
        "model": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42,
        "cv_folds": 5
    })

    for metric, value in logistic_metrics.items():
        mlflow.log_metric(f"CV_{metric}", float(value))

    mlflow.log_artifact(
        str(trained_models_dir / "logistic_regression.joblib"),
        artifact_path="model"
    )

# Updating the model comparison results

model_results.append({
    "Model": "Logistic Regression",
    **logistic_metrics
})

# Displaying model performance

print("Model: Logistic Regression")

print("\nCross-Validation Performance:")

for metric, value in logistic_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrained Model Saved Successfully.")
print("MLflow Run Logged Successfully.")

print("\nModel Comparison:")
display(
    pd.DataFrame(model_results)
    .sort_values("F1_Macro", ascending=False)
    .reset_index(drop=True)
)

Model: Logistic Regression

Cross-Validation Performance:
Accuracy: 0.3612
Precision_Macro: 0.3642
Recall_Macro: 0.3594
F1_Macro: 0.3582

Trained Model Saved Successfully.
MLflow Run Logged Successfully.

Model Comparison:


,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Logistic Regression,0.36125,0.364212,0.359352,0.358197
1,Dummy Classifier,0.35750,0.119167,0.333333,0.175566
